# Traffic Demand Prediction — Modeling

Validation-first workflow. Headline = **H1 temporal-forward R2** (train day 48 -> day 49); H2 = daytime block; H3 = spatial guardrail (unseen geohashes). All target-encodings are out-of-fold (leak-free). See `reports/approach.md` for the full write-up.

In [1]:
import sys; sys.path.append('../src')
import numpy as np, pandas as pd, json
import features as F, pipeline as P, cv
import warnings; warnings.filterwarnings('ignore')
train, test = F.load_raw(); train = P.prepare(train)
print('train', train.shape, '| test', test.shape)

train (77299, 32) | test (41778, 10)


## 1. Baseline ladder (forward day48->day49)
RoadType (stable structure) transfers best; fine geohash/slot memorization is dragged down by day-to-day drift. Confirms the structural core, not memorization, is the foundation.

In [2]:
for name, val in cv.baseline_ladder(train):
    print(f'  {name:20s} forward R2 = {val:.4f}')

  global_mean          forward R2 = -0.0076
  RoadType_mean        forward R2 = 0.7554
  per_geohash_mean     forward R2 = 0.6560
  d48_slot_raw         forward R2 = 0.5223
  d48_slot_affine      forward R2 = 0.6341


## 2. The overfitting fix (the key lesson)
Default LightGBM memorizes day 48 (train R2 ~0.90) but generalizes poorly (H1 0.71). Heavy regularization closes the gap and lifts H1 above the baseline.

In [3]:
best = json.load(open('../models/best_lgbm_params.json'))['params']
fit, val = cv.split_H1(train)
Xtr = P.build_oof(fit); ytr = fit['demand'].to_numpy()
Xva = P.build_fit_transform(fit, val); yva = val['demand'].to_numpy()
import lightgbm as lgb
default = cv.lgbm_factory(); default.fit(Xtr[P.ROBUST_SET], ytr)
tuned = cv.lgbm_factory(best); tuned.fit(Xtr[P.ROBUST_SET], ytr)
for nm, m in [('default', default), ('tuned (Optuna)', tuned)]:
    tr = cv.r2(ytr, np.clip(m.predict(Xtr[P.ROBUST_SET]),0,1))
    va = cv.r2(yva, np.clip(m.predict(Xva[P.ROBUST_SET]),0,1))
    print(f'  {nm:16s} train R2 = {tr:.3f}   H1 = {va:.4f}')

  default          train R2 = 0.899   H1 = 0.7426


  tuned (Optuna)   train R2 = 0.832   H1 = 0.7810


## 3. Segmentation: memorization helps Residential, hurts Highway/Street
Highway is ~71% of total variance but within-Highway forward R2 ~ 0 (unpredictable drift). Within Residential (80% of rows), the geohash signal is stable and helps. So we route by RoadType.

In [4]:
rt = val['RoadType_ord'].to_numpy()
p_rob  = np.clip(tuned.predict(Xva[P.ROBUST_SET]),0,1)
memo_m = cv.lgbm_factory(best); memo_m.fit(Xtr[P.MEMO_SET], ytr)
p_memo = np.clip(memo_m.predict(Xva[P.MEMO_SET]),0,1)
for name,code in [('Residential',0),('Street',1),('Highway',2)]:
    m = rt==code
    print(f'  within-{name:11s} robust R2={cv.r2(yva[m],p_rob[m]):7.3f}  memo R2={cv.r2(yva[m],p_memo[m]):7.3f}')
seg = p_rob.copy(); seg[rt==0] = p_memo[rt==0]
print(f'\n  overall  robust={cv.r2(yva,p_rob):.4f}  full-memo={cv.r2(yva,p_memo):.4f}  SEGMENTED={cv.r2(yva,seg):.4f}')

  within-Residential robust R2=  0.172  memo R2=  0.364
  within-Street      robust R2= -0.171  memo R2= -6.456
  within-Highway     robust R2= -0.014  memo R2= -0.163

  overall  robust=0.7810  full-memo=0.7643  SEGMENTED=0.8051


## 4. Final model metrics (segmented + seed-averaged + calibrated)
Produced by `src/finalize.py`. H1 and H2 land within 0.01 — the signature of a non-overfit model.

In [5]:
fm = json.load(open('../models/final_metrics.json'))
print('FINAL  H1 (forward) raw=%.4f -> calibrated=%.4f' % (fm['H1_raw'], fm['H1_cal']))
print('       H2 (daytime) raw=%.4f -> calibrated=%.4f' % (fm['H2_raw'], fm['H2_cal']))
print('       H3 (spatial guardrail)        = %.4f' % fm['H3'])
print('       calibration a=%.4f b=%.4f' % (fm['a'], fm['b']))
sub = pd.read_csv('../submissions/submission.csv')
print('\nsubmission.csv shape', sub.shape, '| mean pred %.4f' % sub['demand'].mean()); sub.head()

FINAL  H1 (forward) raw=0.8045 -> calibrated=0.8233
       H2 (daytime) raw=0.8175 -> calibrated=0.8247
       H3 (spatial guardrail)        = 0.6396
       calibration a=0.0125 b=1.0178

submission.csv shape (41778, 2) | mean pred 0.1454


,Index,demand
0,0,0.051159
1,1,0.041657
2,2,0.036586
3,3,0.069968
4,4,0.071470


### Conclusion
Honest forward-in-time R2 ≈ **0.823** (expected leaderboard ≈ 82). Built for generalization, not for the leaked public 100s. Every gain (regularization, segmentation, calibration) was validated on two independent held-out axes.